# PACER-Net | 01 Dataset verification
CPU only. Attach both datasets via Add Input. Configure paths and label mappings after inspecting the dataset documentation. External data are audited for integrity only; do not use them to select model settings.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, nibabel as nib
import hashlib, gc
ROOT=Path('/kaggle/input'); OUT=Path('/kaggle/working/pacer_phase1'); OUT.mkdir(parents=True,exist_ok=True)
for p in sorted(ROOT.rglob('*')):
    if p.is_dir() and len(p.relative_to(ROOT).parts)<=4: print(p)


/kaggle/input/datasets
/kaggle/input/datasets/prksh830
/kaggle/input/datasets/prksh830/cptac-pda
/kaggle/input/datasets/prksh830/cptac-pda/Pancreas_segmentations
/kaggle/input/datasets/prksh830/cptac-pda/Tumor_segmentations
/kaggle/input/datasets/prksh830/cptac-pda/Volume files
/kaggle/input/datasets/prksh830/pancreaticcancerdataset
/kaggle/input/datasets/prksh830/pancreaticcancerdataset/Pancreatic Cancer Dataset


In [2]:
# EDIT these paths if automatic discovery fails.
MSD_IMAGES=None; MSD_LABELS=None
CPTAC_IMAGES=None; CPTAC_LABELS=None
# Confirm external label semantics from its documentation before setting True.
CPTAC_LABELS_CONFIRMED=False
CPTAC_PANCREAS_LABELS=(1,2); CPTAC_TUMOR_LABELS=(2,)
# Optional CSV with columns patient_id,image,mask; paths relative to /kaggle/input or absolute.
# Use for mismatched filenames or nested layouts. Never pair by sorted order.
CPTAC_MANIFEST=None

def discover(name):
    hits=[p for p in ROOT.rglob(name) if p.is_dir()]
    if len(hits)!=1: raise ValueError(f'Expected one {name}; found {hits}. Set path manually.')
    return hits[0]
def files(p):
    return sorted(x for x in Path(p).rglob('*') if x.is_file() and (x.name.endswith('.nii') or x.name.endswith('.nii.gz')))
def key(p):
    return p.name.removesuffix('.gz').removesuffix('.nii')
def index(p):
    d={}
    for f in files(p):
        k=key(f)
        if k in d: raise ValueError(f'Duplicate filename ID: {k}')
        d[k]=f
    return d
def pairs(a,b):
    a,b=index(a),index(b)
    return [(k,a.get(k),b.get(k)) for k in sorted(a.keys()|b.keys())]
msd=pairs(MSD_IMAGES or discover('imagesTr'),MSD_LABELS or discover('labelsTr'))
print('MSD candidate IDs:',len(msd))


MSD candidate IDs: 283


In [ ]:
def digest(img):
    # Hash canonical voxel values, shape and geometry; exact duplicates only.
    h=hashlib.sha256(); h.update(str(img.shape).encode()); h.update(np.round(img.affine,5).tobytes())
    for z in range(img.shape[2]): h.update(np.asarray(img.dataobj[:,:,z],dtype='<f4').tobytes())
    return h.hexdigest()

def audit(items, cohort, organ_labels, tumor_labels):
    rows=[]; slices=[]
    for i,(pid,ip,mp) in enumerate(items):
        r=dict(cohort=cohort,patient_id=pid,image=str(ip or ''),mask=str(mp or ''),status='error')
        local=[]
        try:
            if ip is None or mp is None: raise ValueError('Missing image or mask')
            x=nib.as_closest_canonical(nib.load(ip)); m=nib.as_closest_canonical(nib.load(mp))
            if len(x.shape)!=3 or x.shape!=m.shape: raise ValueError('Shape mismatch or non-3D volume')
            if not np.allclose(x.affine,m.affine,atol=1e-3,rtol=0): raise ValueError('Affine mismatch; no automatic alignment')
            spacing=np.linalg.norm(x.affine[:3,:3],axis=0); vv=abs(np.linalg.det(x.affine[:3,:3]))
            if not np.isfinite(vv) or vv<=0: raise ValueError('Invalid voxel geometry')
            counts=np.zeros(2,dtype=np.int64); labels=set(); sums=np.zeros(2); sq=np.zeros(2)
            for z in range(x.shape[2]):
                a=np.asarray(x.dataobj[:,:,z],dtype=np.float32); b=np.asarray(m.dataobj[:,:,z])
                if not np.isfinite(a).all() or not np.isfinite(b).all(): raise ValueError('Nonfinite voxel')
                labels.update(np.unique(b).tolist()); p=np.isin(b,organ_labels); t=np.isin(b,tumor_labels)
                for j,roi in enumerate((p,t)):
                    v=a[roi].astype(np.float64); counts[j]+=v.size; sums[j]+=v.sum(); sq[j]+=np.dot(v,v)
                if p.any(): local.append(dict(cohort=cohort,patient_id=pid,z=z,tumor_present=int(t.any()),pancreas_area_mm2=p.sum()*vv/spacing[2],tumor_area_mm2=t.sum()*vv/spacing[2]))
            if not labels.issubset({0,*organ_labels,*tumor_labels}): raise ValueError(f'Unexpected labels {labels}')
            if not counts[0]: raise ValueError('Empty pancreas')
            r.update(status='ok',shape=str(x.shape),spacing_x=spacing[0],spacing_y=spacing[1],spacing_z=spacing[2],labels=str(sorted(labels)),pancreas_ml=counts[0]*vv/1000,tumor_ml=counts[1]*vv/1000,tumor_ratio=counts[1]/counts[0],pancreas_hu_mean=sums[0]/counts[0],pancreas_hu_sd=np.sqrt(max(0,sq[0]/counts[0]-(sums[0]/counts[0])**2)),tumor_hu_mean=sums[1]/counts[1] if counts[1] else np.nan,positive_slices=sum(s['tumor_present'] for s in local),negative_slices=sum(1-s['tumor_present'] for s in local),image_hash=digest(x))
            slices.extend(local)
        except Exception as e: r.update(status='error',error=str(e))
        rows.append(r)
        if (i+1)%10==0: print(cohort,i+1,'/',len(items))
    d=pd.DataFrame(rows); s=pd.DataFrame(slices)
    d.to_csv(OUT/f'{cohort}_Dataset_Statistics.csv',index=False); s.to_csv(OUT/f'{cohort}_Slices.csv',index=False)
    print(d.status.value_counts()); return d,s
msd_stats,msd_slices=audit(msd,'MSD',(1,2),(2,))


MSD 10 / 283
MSD 20 / 283
MSD 30 / 283
MSD 40 / 283
MSD 50 / 283
MSD 70 / 283
MSD 80 / 283
MSD 90 / 283


In [ ]:
# External audit: run after confirming label mapping and CT/mask pairing.
if not CPTAC_LABELS_CONFIRMED:
    print('STOP: confirm CPTAC label mapping in configuration cell before external audit.')
else:
    if CPTAC_MANIFEST:
        df=pd.read_csv(CPTAC_MANIFEST,dtype={'patient_id':str}); assert df.patient_id.is_unique
        ext=[(r.patient_id,ROOT/str(r.image),ROOT/str(r.mask)) for r in df.itertuples()]
    else:
        assert CPTAC_IMAGES and CPTAC_LABELS, 'Set external image and mask folders or manifest'
        ext=pairs(CPTAC_IMAGES,CPTAC_LABELS)
    ext_stats,ext_slices=audit(ext,'CPTAC',CPTAC_PANCREAS_LABELS,CPTAC_TUMOR_LABELS)


In [ ]:
# Duplicate and completeness report; exact hashes cannot establish patient independence.
frames=[pd.read_csv(p) for p in OUT.glob('*_Dataset_Statistics.csv')]
all_stats=pd.concat(frames,ignore_index=True)
valid=all_stats[all_stats.status=='ok'].copy()
if 'image_hash' in valid:
    dup=valid[valid.image_hash.notna() & valid.image_hash.duplicated(keep=False)]
    dup.to_csv(OUT/'Exact_Duplicate_Images.csv',index=False)
    print('Exact duplicate rows:',len(dup))
print(all_stats.groupby(['cohort','status']).size())
all_stats[all_stats.status!='ok'].to_csv(OUT/'Integrity_Errors.csv',index=False)
print('Expected counts: MSD 281; CPTAC 42. Verify patient identity, provenance and annotation mapping manually.')
